# 02. Workspace backend

Добавляем backend. По умолчанию это безопасный workspace-scoped filesystem backend.
Shell включается только через `OPENCLAW_ENABLE_LOCAL_SHELL=1`.

In [ ]:
from __future__ import annotations

import os
from pathlib import Path

from deepagents import create_deep_agent
from dotenv import load_dotenv

load_dotenv()

DEFAULT_MODEL = "openai:gpt-5.3-codex"

def model_name() -> str:
    return os.getenv("OPENCLAW_MODEL", DEFAULT_MODEL)

In [ ]:
BASE_PROMPT = """\
You are OpenClaw, an experimental open-source coding agent built with LangChain
Deep Agents. You help with software engineering, repository navigation, product
research, and implementation.

Work like a careful senior engineer:
- inspect before editing;
- keep changes focused;
- verify with tests or equivalent checks;
- explain only the useful result to the user.

If local shell execution is unavailable, use filesystem tools and clearly state
which verification would require shell access.
"""

In [ ]:
def workspace_root() -> Path:
    return Path(os.getenv("OPENCLAW_WORKSPACE", ".")).expanduser().resolve()


def backend():
    """Filesystem by default; local shell only when explicitly enabled."""
    root_dir = workspace_root()

    if os.getenv("OPENCLAW_ENABLE_LOCAL_SHELL") != "1":
        from deepagents.backends import FilesystemBackend

        return FilesystemBackend(root_dir=root_dir, virtual_mode=True)

    from deepagents.backends import LocalShellBackend

    return LocalShellBackend(
        root_dir=root_dir,
        virtual_mode=True,
        inherit_env=True,
        timeout=120,
        max_output_bytes=80_000,
    )

In [ ]:
agent = create_deep_agent(
    model=model_name(),
    tools=[],
    system_prompt=BASE_PROMPT,
    backend=backend(),
    interrupt_on={"execute": True},
)

In [ ]:
# Smoke check: the graph object is created locally.
print(type(agent).__name__)

Demo prompt: "Inspect the workspace files and summarize what this project does".